In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report
from sklearn.svm import SVC
from sklearn import tree
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)
import tensorflow.keras as keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import visualkeras
import tensorflow as tf
tf.data.experimental.enable_debug_mode()
from tensorflow.keras.callbacks import ModelCheckpoint
from tqdm.notebook import tqdm_notebook
from sklearn.tree import export_graphviz
from collections import defaultdict


In [5]:
# this function also calculates the second most frequent

def sequence_mining(team, opponent, df):
    combined_df = pd.read_csv(df)
    combined_df = combined_df.replace({str(team):'same', str(opponent):'other'}, regex=True)

    df = pd.DataFrame()
    encoders = []

    for column in combined_df.columns[:-1]:
        le = LabelEncoder()
        encoders.append(le)
        df[column] = le.fit_transform(combined_df[column])

    df = pd.concat([df, combined_df.iloc[:, -1]], axis=1)

    undersample_len = len(df[df['class'] == 1])

    undersample_df = df[df['class'] == 0].sample(n=undersample_len, random_state=43)
    df = pd.concat([df[df['class'] == 1], undersample_df])

    events_idx = {}

    sequence_mining_html = ""

    # Total number of runs (sequences)
    total_sequences = len(combined_df[combined_df['class'] == 1])

    # Prepare lists to store pattern lengths and their corresponding max counts (frequencies)
    pattern_lengths = []
    frequencies = []

    freqs = []

    for j, event in zip(range(12, 112, 11), range(10, 0, -1)):

        event_dict = {}

        a = combined_df.iloc[:, -j:-1][combined_df['class'] == 1]

        # Count occurrences of each row
        row_counts = defaultdict(int)
        for i in range(len(a)):
            row_tuple = tuple(a.iloc[i])
            row_counts[row_tuple] += 1

        # Find the rows with the maximum and second maximum counts
        sorted_row_counts = sorted(row_counts.items(), key=lambda x: x[1], reverse=True)
        mc_row, max_count = sorted_row_counts[0]
        sc_row, second_max_count = sorted_row_counts[1] if len(sorted_row_counts) > 1 else (None, 0)

        # Find all indices of the rows that match the row with the maximum count
        mc_indices = a.apply(lambda row: tuple(row) == mc_row, axis=1)
        mc_indices = mc_indices[mc_indices].index.tolist()

        # Calculate the ratios of the max count and second max count to total sequences
        max_count_ratio = max_count / total_sequences
        second_max_count_ratio = second_max_count / total_sequences if second_max_count > 0 else 0

        events_idx[event] = mc_indices

        # Store the pattern length (event) and frequency (max_count) for ideal length calculation
        pattern_lengths.append(event)
        frequencies.append(max_count)

        # Add sequence mining results to HTML
        sequence_mining_html += f"<p><strong>Last {abs(event-11)} events before run</strong></p>"
        sequence_mining_html += f"<p>Max Count: {max_count}</p>"
        sequence_mining_html += f"<p>Ratio of Max Count to Total Sequences: {max_count_ratio:.2%}</p>"
        sequence_mining_html += f"<p>Second Max Count: {second_max_count}</p>"
        sequence_mining_html += f"<p>Ratio of Second Max Count to Total Sequences: {second_max_count_ratio:.2%}</p>"
        sequence_mining_html += f"<table class='table table-striped'>{combined_df.iloc[mc_indices[0], -j:-1].to_frame().dropna().T.to_html()}</table>"    

        event_dict['Event'] = abs(event-11)
        event_dict['Frequency'] = max_count
        event_dict['Ratio'] = np.round(max_count_ratio, 4)
        event_dict['Sec Frequency'] = second_max_count
        event_dict['Sec Ratio'] = np.round(second_max_count_ratio, 4)

        freqs.append(event_dict)

    m = pd.DataFrame(freqs)

    return m


sequence_mining('away','home','ATL_runs.csv')

C:\Users\gsevr\AppData\Local\Temp\ipykernel_81944\1005350902.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[column] = le.fit_transform(combined_df[column])
C:\Users\gsevr\AppData\Local\Temp\ipykernel_81944\1005350902.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[column] = le.fit_transform(combined_df[column])
C:\Users\gsevr\AppData\Local\Temp\ipykernel_81944\1005350902.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

,Event,Frequency,Ratio,Sec Frequency,Sec Ratio
0,1,155,0.2763,64,0.1141
1,2,31,0.0553,25,0.0446
2,3,13,0.0232,11,0.0196
3,4,8,0.0143,7,0.0125
4,5,4,0.0071,4,0.0071
5,6,4,0.0071,2,0.0036
6,7,2,0.0036,2,0.0036
7,8,1,0.0018,1,0.0018
8,9,1,0.0018,1,0.0018
9,10,1,0.0018,1,0.0018
